[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/DualScoreMatching/blob/main/tp/auteurs/affine_and_sparsity_colab.ipynb)

**Notebook des auteurs** (Guth, Kadkhodaie, Simoncelli, 2025), adapté pour Colab et pour un ordinateur sans GPU par le TP « score matching dual » : les cellules marquées « TP » sont des ajouts ; le reste est inchangé.

Par défaut, grille 21×21 ; 101×101 avec les réglages des auteurs (GPU conseillé).

**Sur Colab** : `Exécution > Modifier le type d'exécution > GPU T4`, puis exécuter les cellules dans l'ordre. Réglages réduits par défaut, même avec un GPU ; réglages des auteurs (non vérifié) : `if False:` -> `if True:` dans la cellule de compatibilité ; données : même changement dans la cellule « Données » (téléchargement de 900 Mo).

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "DualScoreMatching"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/*.py',
            '/networks/',
            '/tp/tp_outils.py',
            '/models/finalclean_imagenet64_grayscale_lr0.0005_1Msteps10decays/',
            '/outputs/energies_test.pt']

import os, sys, subprocess
if "google.colab" in sys.modules:
    root = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(root, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), root], check=True)
    subprocess.run(["git", "-C", root, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", root, "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'einops', 'PyWavelets'], check=True)
    os.chdir(os.path.join(root, ""))
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
    os.chdir(os.path.join(racine, ""))
sys.path.insert(0, os.getcwd())
print("Répertoire courant :", os.getcwd())

In [ ]:
# Compatibilité (ajout du TP) : le code des auteurs a été écrit pour un GPU et une version de PyTorch antérieure
# à 2.6 (torch.load lisait alors par défaut des objets Python, et non seulement des poids).
import functools, sys, torch
sys.argv = sys.argv[:1]   # main.parse_args lit la ligne de commande : celle du noyau Jupyter n'est pas la sienne
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not getattr(torch.load, "_tp", False):
    _torch_load = torch.load
    def _load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)               # points de contrôle des auteurs (dépôt de confiance)
        if not torch.cuda.is_available():
            kwargs.setdefault("map_location", "cpu")           # tenseurs enregistrés sur GPU
        return _torch_load(*args, **kwargs)
    _load._tp = True
    torch.load = _load
import networks.model as _nm
if not torch.cuda.is_available() and not getattr(_nm.Model.__init__, "_tp", False):
    # sans GPU, Model n'enveloppe pas le réseau dans DataParallel, alors que main.py lit model.network.module
    _init = _nm.Model.__init__
    def _init_cpu(self, network, *args, **kwargs):
        _init(self, network, *args, **kwargs)
        self.network = torch.nn.DataParallel(self.network)    # sans GPU, DataParallel appelle le réseau tel quel
    _init_cpu._tp = True
    _nm.Model.__init__ = _init_cpu
os.makedirs("tp/resultats/auteurs", exist_ok=True)   # écritures du notebook (au lieu de outputs/, laissé intact)
# Réglages : réduits par défaut, même avec un GPU (le calcul se fait alors sur le GPU) ; réglages des auteurs (non
# vérifié, GPU indispensable) : remplacer False par True ci-dessous.
REGLAGES_AUTEURS = False
if False:
    REGLAGES_AUTEURS = torch.cuda.is_available()
print("calcul sur", DEVICE, "- réglages", "des auteurs" if REGLAGES_AUTEURS else "réduits")

In [ ]:
# Données (ajout du TP ; une fois par machine, sur Colab à chaque nouvelle session) : ImageNet64 n'est pas distribué
# par les auteurs. On le reconstruit au format attendu par data.py à partir d'un miroir HuggingFace de la même version
# (Chrabaszcz et al., 2017) : validation complète (50 000 images, 417 Mo, dans l'ordre des auteurs) et premier fragment
# d'entraînement (58 000 images, 483 Mo). Rien n'est téléchargé par défaut : remplacer False par True ci-dessous.
if False:
    import sys
    sys.path.insert(0, "tp")
    import tp_outils
    tp_outils.preparer_imagenet64_auteurs()

# Affine transformations and sparsity

In [ ]:

import numpy as np
from matplotlib import pyplot as plt
import torch

from tqdm.auto import tqdm, trange

from main import *

params = {
      'text.usetex': False,  # TP : pas de LaTeX sur Colab
      'font.family': 'sans serif'
}
matplotlib.rcParams.update(params)

## Load ImageNet64 model

In [ ]:
def load_exp(name, step="last", log=True, dataloaders=False):
    """ Load an experiment with a given name. step can be an integer, "best", or "last" (default). """
    exp_dir = Path("models") / name

    with open(exp_dir / "args.json") as f:
        args_dict = json.load(f)
    args_dict["dir"] = "models"  # TP : args.json indique "experiments"

    ctx = TrainingContext(**args_dict, step=step, key_remap=None, seed=None, dataloaders=dataloaders, writer=False)
    if log:
        print(f"{name}: retrieved model at step {ctx.step} and test loss {ctx.test_perf.loss:.2e}")

    # Disable DataParallel (needed for Hessian computation)
    ctx.model.network = ctx.model.network.module

    # Put in eval mode and disable gradients with respect to all parameters.
    ctx.model.eval()
    for p in ctx.model.parameters():
        p.requires_grad = False

    return ctx

ctx = load_exp("finalclean_imagenet64_grayscale_lr0.0005_1Msteps10decays", dataloaders=True)
ctx.model.log_normalization_constant = 5.845e+03  # Calculated from energy_histogram.ipynb
device = ctx.device
dataset_info = ctx.dataset_info
d = dataset_info.dimension
# Dataloaders have horizontal flips.
train_dataloader_grayscale = ctx.new_dataloader(train=True, batch_size=1024)
test_dataloader_grayscale = ctx.new_dataloader(train=False, batch_size=1024)

c = lambda e: LogTensor(e, d).to(base="dBs", sign="logp", per_dimension=True)

## Reference images

In [ ]:
import torchvision.transforms.v2 as v2
from PIL import Image

n = 15
colormap = "plasma"
colors = matplotlib.colormaps.get_cmap(colormap)(np.linspace(0, 1, n))[:, :3]  # (N, 3)

color_transforms = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
grayscale_transforms = v2.Compose([v2.Grayscale(), v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
apply = lambda t, x: torch.stack([t(Image.fromarray(xx)) for xx in x], dim=0)

idx_ref = np.asarray([21143, 34295, 42627, 47510,  8718,  4375,  1266, 47346, 11592, 36426, 31424, 17305, 14457, 39755, 41191])
imgs_ref = test_dataloader_grayscale.dataset.data[idx_ref]  # (N, H, W, C) uint8 numpy array
print(type(imgs_ref), imgs_ref.shape)
imgs_ref_color = apply(color_transforms, imgs_ref).to(DEVICE)
imgs_ref_grayscale = apply(grayscale_transforms, imgs_ref).to(DEVICE)
print(tensor_summary_stats(imgs_ref_color), tensor_summary_stats(imgs_ref_grayscale), sep="\n")
plot_img(rearrange(imgs_ref_color, "n c h w -> 1 h n w c"), inches_per_pixel=0.025)
plot_img(rearrange(imgs_ref_grayscale, "n c h w -> 1 h n w c"), inches_per_pixel=0.025, cmap="gray")

## Probability under affine transforms

In [ ]:
m = 101 if REGLAGES_AUTEURS else 21  # TP : grille réduite par défaut
alphas = torch.linspace(0, 1, m).to(DEVICE)  # (A,)
betas = torch.linspace(0, 1, m).to(DEVICE)  # (B,)
energies_ids = {}  # img_id -> (B, A)

for i in range(n):
    x = imgs_ref_grayscale[i, None] * alphas[:, None, None, None]  # (A, C, H, W)
    energies = []
    for b in betas:
        output = ctx.model(model_input(x + b), compute_scores=False)
        energies.append(output.energy.cpu())  # (A,)

    energies_ids[i] = torch.stack(energies)  # (B, A)

In [ ]:
colors = plt.cm.plasma(np.linspace(0, 1, n))

plt.figure(figsize=(3, 2.5), layout="constrained")
for i in range(n):
    plt.plot(alphas.cpu()[:m//2 + 1], c(energies_ids[i])[m//2, :m//2 + 1], color=colors[i])
eps = 0.02
plt.xlim(-eps, 0.5 + eps)
plt.xlabel('$a$')
plt.xticks([0, 0.1, 0.2, 0.3, 0.4, 0.5])
plt.ylim(0, 50)
plt.ylabel(r'$\log p_\theta(ax + 1/2)$')
plt.savefig("tp/resultats/auteurs/contrast_energy.pdf", transparent=True, bbox_inches="tight", pad_inches=0)
plt.show()

In [6]:
i = n // 2
mask = 1 - torch.triu(torch.ones_like(energies_ids[i]), diagonal=1)
mask = torch.triu(torch.ones_like(energies_ids[i]), diagonal=1)
energies_ids_n = c(energies_ids[i]).flip(dims=[0])  # Reverse vertical axis
energies_ids_n[mask == 1] = np.nan

In [ ]:
f, axs = plt.subplots(1, 1, figsize=(2.5, 2.5), layout="constrained")
im = axs.imshow(energies_ids_n, cmap="hot")
axs.set_xlabel('$a$')
axs.set_ylabel('$b$')
f.colorbar(im, ax=axs, fraction=0.05, ticks=[-10, 0, 10, 20, 30, 40, 50], label=r"$\log p_\theta(ax + b)$")
axs.set_xticks(range(0, len(alphas), 20))
axs.set_xticklabels(np.round(alphas[::20].cpu().numpy(), 2), rotation=45)
axs.set_yticks(range(0, len(betas), 20))
axs.set_yticklabels(np.round(betas[::20].cpu().flip(dims=[0]).numpy(), 2))
axs.spines['top'].set_visible(False)
axs.spines['right'].set_visible(False)
plt.savefig("tp/resultats/auteurs/energy_affine.pdf", transparent=True, bbox_inches="tight", pad_inches=0)
plt.show()

## Probability and sparsity

In [8]:
# Load images from test set
import torchvision.transforms.v2 as v2
from PIL import Image
grayscale_transforms = v2.Compose([v2.Grayscale(), v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
apply = lambda t, x: torch.stack([t(Image.fromarray(xx)) for xx in x], dim=0)
imgs = test_dataloader_grayscale.dataset.data  # (N, H, W, C) uint8 numpy array
imgs = apply(grayscale_transforms, imgs)

# Load energies of imagenet test set
_, energies = torch.load("outputs/energies_test.pt")

In [ ]:
# Compute wavelet coefficients
import pywt

def multi_scale_decompose(images, J, device, wavelet='db1', mode='symmetric', orth_forward=True):
    """ multi scale wavelet transform
    @J: number of scales
    @images: numpy array of size (*,1, L, L)
    Returns a list (length J) of tensor coeffs, each with 4 channels (low pass + 3 detail bands).
    """
    all_coeffs = []
    if torch.is_tensor(images) and images.is_cuda:
        low = images.cpu()
    else:
        low = images

    for j in range(J):
        low, high = pywt.dwt2(low, wavelet, mode)
        coeffs = np.concatenate((low,) + high, axis=-3)  # (*, 4, L/2, L/2)
        if orth_forward:
            coeffs = coeffs / 2**(j+1)
        all_coeffs.append(torch.from_numpy(coeffs).to(device=device, dtype=torch.float32))

    return all_coeffs

coeffs = multi_scale_decompose(images=imgs, J=6, device=device, wavelet='db1', mode='symmetric', orth_forward=True)

In [10]:
# Compute participation ratios
prs = []
for n in range(imgs.shape[0]):
    temp = []
    for j in range(len(coeffs)):  # Leave out low pass at each scale except for the last scale
        if j < 5:
            temp.append(coeffs[j][n][1:4].flatten())
        else:
            temp.append(coeffs[j][n][0:4].flatten())

    coeffs_im = torch.concatenate(temp)
    prs.append((coeffs_im.norm(1)**2 / coeffs_im.norm(2)**2 /(coeffs_im.shape[0])).cpu() )
prs = torch.tensor(prs)

In [ ]:
plt.figure(figsize=(3, 2.5), layout="constrained")
plt.hist2d(prs.numpy(), c(energies).cpu().numpy(), bins=100, range=((0, 0.6), (0, 30)), cmap="hot", rasterized=True)
plt.ylabel("Log probability (dB/dim)")
plt.xlabel("Participation ratio")
slope, intercept = np.polyfit(prs, c(energies).cpu(), 1)
x = np.asarray([0, 0.6])
y_fit = slope * x + intercept
plt.plot(x, y_fit, color='white', alpha=0.5)
plt.savefig("tp/resultats/auteurs/energy_sparsity.pdf", transparent=True, bbox_inches="tight", pad_inches=0, dpi=600)
plt.show()